# Inference — Italian WITS

Generates summaries on Italian WITS test data using SigExt + Llama-3.1.

**Memory strategy**: SigExt runs on CPU -> unloaded -> LLM on GPU.

In [ ]:
!uv pip install -e ../..

In [ ]:
from huggingface_hub import login
login()

## Configuration

In [ ]:
from sm_sip.config import SigExtConfig, InferenceConfig

sigext_config = SigExtConfig.from_preset("it", "xlmr-5k-060t")
inference_config = InferenceConfig(
    lang="it",
    quantization="8bit",
    prompt_type="source_aware",
    num_test_samples=100,
)

## Step 1: Preprocess with SigExt (CPU)

In [ ]:
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, unload_sigext_model, preprocess_dataset

test_data = get_test_data(
    lang="it",
    num_samples=inference_config.num_test_samples,
    skip_samples=sigext_config.skip_samples,
)

sigext_model, sigext_tokenizer = load_sigext_model(sigext_config.model_id, device="cpu")
processed_data = preprocess_dataset(test_data, sigext_model, sigext_tokenizer, lang="it")
unload_sigext_model(sigext_model, sigext_tokenizer)
print(f"Preprocessed {len(processed_data)} samples.")

## Step 2: Generate Summaries

In [ ]:
from sm_sip.models import load_llm, create_summary_chain
from sm_sip.prompts import get_summary_prompt
from sm_sip.pipelines import run_inference

llm_model, llm_tokenizer, gen_pipe = load_llm(
    inference_config.llm_model_id,
    inference_config.quantization,
    seed=inference_config.seed,
)
summary_chain = create_summary_chain(gen_pipe, get_summary_prompt("it", inference_config.prompt_type))
results = run_inference(processed_data, summary_chain)

## Save & Cleanup

In [ ]:
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory
from datetime import datetime

save_results({
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "sigext_model": sigext_config.model_id,
        "llm_model": inference_config.llm_model_id,
        "quantization": inference_config.quantization,
        "prompt_type": inference_config.prompt_type,
        "seed": inference_config.seed,
        "num_samples": len(results),
    },
    "samples": results,
}, f"results/italian/inference_{inference_config.prompt_type}.json")

del llm_model, llm_tokenizer, gen_pipe
clear_gpu_memory()
print("Done!")